# EXP 02 — Leakage-Safe Historical Strength Engine, Frozen vs Recursive Future Simulation, dan CatBoost Decomposition + Joint Decoder

Eksperimen ini adalah lanjutan dari baseline shared-feature pada EXP 01.  
Fokus utamanya bukan mengganti keluarga model, tetapi **menambahkan representasi dynamic strength yang legal, leakage-safe, dan tetap feasible untuk horizon test yang panjang**.

Di notebook ini kita membandingkan tiga variant utama:

1. **control_static** — static shared features ala EXP 01,
2. **history_frozen** — static + historical strength, tetapi performance state dibekukan sejak cutoff,
3. **history_recursive** — static + historical strength, lalu future state di-update secara pseudo-recursive memakai prediksi model sendiri.

Metrik evaluasi utama tetap **offline AW-MAE**, sehingga seluruh keputusan model terbaik akan diambil berdasarkan skor tersebut.


## 01. Setup, Seed, dan Konfigurasi Path


In [11]:
import os
import json
import math
import copy
import random
import warnings
from pathlib import Path
from collections import Counter, deque, defaultdict
from itertools import product

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from catboost import CatBoostRegressor, CatBoostClassifier, Pool
from sklearn.metrics import classification_report

print("[INFO] Semua library berhasil di-import.")


[INFO] Semua library berhasil di-import.


In [12]:
# === KONFIGURASI GLOBAL ===
SEED = 42

TRAIN_PATH = "../data/train.csv"
TEST_PATH = "../data/test.csv"
SAMPLE_SUB_CANDIDATES = [
    "../data/sample submission.csv",
    "../data/sample_submission.csv",
]
META_PATH = "../data/metadata.txt"

OUT_ROOT = "../outputs/exp02_history_strength_engine_recursive"
FIG_DIR = f"{OUT_ROOT}/figures"
PRED_DIR = f"{OUT_ROOT}/predictions"
SUB_DIR = f"{OUT_ROOT}/submissions"
SUM_DIR = f"{OUT_ROOT}/summaries"

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 40)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
warnings.filterwarnings("ignore")

print(f"[INFO] SEED = {SEED}")


[INFO] SEED = 42


In [13]:
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"[INFO] Seed set to {seed}")

seed_everything(SEED)


[INFO] Seed set to 42


In [14]:
def resolve_existing_path(candidates):
    for path in candidates:
        if Path(path).exists():
            return path
    return candidates[0]

SAMPLE_SUB_PATH = resolve_existing_path(SAMPLE_SUB_CANDIDATES)
print(f"[INFO] Sample submission path yang dipakai: {SAMPLE_SUB_PATH}")

for d in [FIG_DIR, PRED_DIR, SUB_DIR, SUM_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)
    print(f"[OK] {d}")


[INFO] Sample submission path yang dipakai: ../data/sample submission.csv
[OK] ../outputs/exp02_history_strength_engine_recursive/figures
[OK] ../outputs/exp02_history_strength_engine_recursive/predictions
[OK] ../outputs/exp02_history_strength_engine_recursive/submissions
[OK] ../outputs/exp02_history_strength_engine_recursive/summaries


### Catatan Setup

Notebook ini dibuat **standalone**, sehingga seluruh helper penting dibawa ulang ke file ini.  
Seluruh path dibuat eksplisit relatif terhadap folder `notebook/`, agar notebook tetap konsisten dengan struktur project kompetisi.


## 02. Validasi File Input


In [15]:
FILE_PATHS = {
    "train": TRAIN_PATH,
    "test": TEST_PATH,
    "sample_submission": SAMPLE_SUB_PATH,
    "metadata": META_PATH,
}

def validate_input_files(file_paths: dict) -> None:
    missing = []
    for name, path in file_paths.items():
        if not Path(path).exists():
            missing.append((name, path))
    if missing:
        msg = "File TIDAK ditemukan:\n" + "\n".join(
            f"  - {name}: {path}" for name, path in missing
        )
        raise FileNotFoundError(msg)
    print("[OK] Semua file input ditemukan.")

validate_input_files(FILE_PATHS)


[OK] Semua file input ditemukan.


## 03. Load Data dan Helper Dasar dari EXP 00/01


In [16]:
# --- Load data ---
train = pd.read_csv(TRAIN_PATH, parse_dates=["date"])
test = pd.read_csv(TEST_PATH, parse_dates=["date"])
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

with open(META_PATH, "r", encoding="utf-8") as f:
    metadata_text = f.read()

print(f"Train : {train.shape}")
print(f"Test  : {test.shape}")
print(f"Sample: {sample_sub.shape}")
print()
print("[INFO] Metadata preview:")
print(metadata_text[:800])


Train : (78772, 47)
Test  : (42422, 20)
Sample: (42422, 3)

[INFO] Metadata preview:
1. Identitas & Info Dasar
Id: Identitas unik untuk setiap baris (Format: match_id_nama_tim).
match_id: ID unik untuk satu pertandingan (satu pertandingan memiliki dua baris Id untuk masing-masing tim).
date: Tanggal pertandingan dilaksanakan.
gender: Jenis kelamin pemain (M untuk Pria, W untuk Wanita).
team: Nama tim utama.
opponent: Nama tim lawan.

2. Kondisi Pertandingan
is_home: Binary (1/0), apakah tim bermain di kandang sendiri.
neutral: Binary (1/0), apakah pertandingan dimainkan di tempat netral.
tournament: Nama kompetisi atau jenis turnamen (misal: Friendly, FIFA World Cup, dll).
venue_country: Negara tempat pertandingan berlangsung.
confederation_team/opp: Konfederasi sepak bola tim/lawan (misal: UEFA, CAF, CONMEBOL).

3. Metrik Performa (Hanya tersedia lengkap di train.csv)
elo


In [17]:
# ============================================================
# HELPER FUNCTIONS — ditulis ulang agar notebook standalone
# ============================================================

EXACT_PENALTY = 0.30
OUTCOME_PENALTY = 0.25
GD_PENALTY = 0.15
WRONG_OUTCOME_MULTIPLIER = 1.50
NONLINEAR_POWER = 1.50

DEFAULT_ELO = 1500.0
DEFAULT_ELO_GD = 0.0
DEFAULT_EWM_POINTS = 1.0
DEFAULT_EWM_GF = 1.2
DEFAULT_EWM_GA = 1.2
DEFAULT_EWM_GD = 0.0

ELO_K = 24.0
GD_K = 6.0
EWM_ALPHA = 0.35
HOME_BONUS_ELO = 60.0
HOME_BONUS_GD = 10.0

def summarize_dataframe(df: pd.DataFrame, name: str) -> pd.DataFrame:
    summary = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "missing": df.isna().sum().values,
        "missing_pct": (df.isna().mean() * 100).values,
        "nunique": df.nunique(dropna=True).values,
    }).sort_values(["missing_pct", "column"], ascending=[False, True]).reset_index(drop=True)
    print(f"[INFO] Summary untuk {name}: {df.shape[0]:,} rows x {df.shape[1]:,} cols")
    return summary

def _safe_log1p(series: pd.Series) -> pd.Series:
    x = pd.to_numeric(series, errors="coerce")
    x = x.where(x >= 0)
    return np.log1p(x)

def _safe_divide(num: pd.Series, den: pd.Series) -> pd.Series:
    num = pd.to_numeric(num, errors="coerce")
    den = pd.to_numeric(den, errors="coerce")
    out = pd.Series(np.nan, index=num.index, dtype=float)
    mask = den.notna() & (den != 0)
    out.loc[mask] = num.loc[mask] / den.loc[mask]
    out.replace([np.inf, -np.inf], np.nan, inplace=True)
    return out

def _safe_mean(values, default=np.nan):
    vals = [v for v in values if pd.notna(v)]
    return float(np.mean(vals)) if len(vals) > 0 else default

def _points_from_score(goals_for: int, goals_against: int) -> int:
    if goals_for > goals_against:
        return 3
    if goals_for == goals_against:
        return 1
    return 0

def _result_indicator(goals_for: int, goals_against: int) -> int:
    if goals_for > goals_against:
        return 1
    if goals_for == goals_against:
        return 0
    return -1

def _canonical_pair_key(gender: str, team_a: str, team_b: str):
    return (str(gender), str(team_a), str(team_b))

def _team_state_key(gender: str, team: str):
    return (str(gender), str(team))

def build_match_level(df: pd.DataFrame, is_train: bool) -> pd.DataFrame:
    """Konversi row-level → match-level.
    Canonicalization: team_a = alfabet pertama; tie-break = Id.
    """
    required_cols = {"match_id", "team", "date"}
    missing_required = required_cols - set(df.columns)
    if missing_required:
        raise KeyError(f"Kolom wajib hilang untuk canonicalization: {missing_required}")

    counts = df.groupby("match_id").size()
    bad = counts[counts != 2]
    if len(bad) > 0:
        raise ValueError(f"{len(bad)} match_id tanpa tepat 2 rows")

    sort_cols = ["match_id", "team"]
    if "Id" in df.columns:
        sort_cols.append("Id")

    df_sorted = df.sort_values(sort_cols).reset_index(drop=True)
    row_a = df_sorted.iloc[0::2].reset_index(drop=True)
    row_b = df_sorted.iloc[1::2].reset_index(drop=True)

    if not np.array_equal(row_a["match_id"].values, row_b["match_id"].values):
        raise ValueError("Pairing row_a dan row_b tidak sejajar setelah sort canonical.")

    result = pd.DataFrame()
    shared_match_cols = ["match_id", "date", "gender", "tournament", "venue_country", "neutral"]
    for col in shared_match_cols:
        if col in df.columns:
            result[col] = row_a[col].values

    for col in ["altitude_venue", "temperature_venue"]:
        if col in df.columns:
            result[col] = pd.to_numeric(row_a[col], errors="coerce").values

    result["team_a"] = row_a["team"].astype(str).values
    result["team_b"] = row_b["team"].astype(str).values

    result["team_a_is_home"] = pd.to_numeric(row_a.get("is_home", 0), errors="coerce").fillna(0).astype(float).values
    result["team_b_is_home"] = pd.to_numeric(row_b.get("is_home", 0), errors="coerce").fillna(0).astype(float).values

    mapping = {
        "confederation_team": "confederation",
        "population_team": "population",
        "gdp_per_capita_team": "gdp_per_capita",
        "distance_travel_team": "distance_travel",
    }
    for src, dst in mapping.items():
        if src in df.columns:
            result[f"team_a_{dst}"] = pd.to_numeric(row_a[src], errors="coerce").values
            result[f"team_b_{dst}"] = pd.to_numeric(row_b[src], errors="coerce").values

    if "confederation_team" in df.columns:
        result["team_a_confederation"] = row_a["confederation_team"].astype(str).values
        result["team_b_confederation"] = row_b["confederation_team"].astype(str).values

    if "Id" in df.columns:
        result["row_id_a"] = row_a["Id"].values
        result["row_id_b"] = row_b["Id"].values

    if is_train:
        if not {"team_goals", "opp_goals"}.issubset(df.columns):
            raise KeyError("Train match-level membutuhkan kolom team_goals dan opp_goals.")
        result["team_a_goals"] = pd.to_numeric(row_a["team_goals"], errors="coerce").astype(float).values
        result["team_b_goals"] = pd.to_numeric(row_b["team_goals"], errors="coerce").astype(float).values

    return result

def match_predictions_to_submission(
    test_row_df: pd.DataFrame,
    pred_match_df: pd.DataFrame,
    canonical_team_a_col: str = "team_a",
    canonical_team_b_col: str = "team_b",
    pred_a_col: str = "pred_team_a_goals",
    pred_b_col: str = "pred_team_b_goals",
) -> pd.DataFrame:
    """Konversi prediksi match-level canonical → submission row-level."""
    required_pred_cols = {"match_id", canonical_team_a_col, canonical_team_b_col, pred_a_col, pred_b_col}
    missing_pred = required_pred_cols - set(pred_match_df.columns)
    if missing_pred:
        raise KeyError(f"Kolom prediksi kurang: {missing_pred}")

    base = test_row_df[["Id", "match_id", "team", "opponent"]].copy()
    lookup = pred_match_df[["match_id", canonical_team_a_col, canonical_team_b_col, pred_a_col, pred_b_col]].copy()
    merged = base.merge(lookup, on="match_id", how="left", validate="many_to_one")

    if merged[[pred_a_col, pred_b_col]].isna().any().any():
        n_bad = int(merged[[pred_a_col, pred_b_col]].isna().any(axis=1).sum())
        raise ValueError(f"Ada {n_bad} row test yang belum mendapatkan prediksi match-level.")

    team_is_a = merged["team"].astype(str) == merged[canonical_team_a_col].astype(str)
    team_is_b = merged["team"].astype(str) == merged[canonical_team_b_col].astype(str)

    if not (team_is_a | team_is_b).all():
        raise ValueError("Ada row test yang team-nya tidak cocok dengan pasangan canonical team_a/team_b.")

    merged["team_goals"] = np.where(team_is_a, merged[pred_a_col], merged[pred_b_col])
    merged["opp_goals"] = np.where(team_is_a, merged[pred_b_col], merged[pred_a_col])

    submission = merged[["Id", "team_goals", "opp_goals"]].copy()
    submission["team_goals"] = pd.to_numeric(submission["team_goals"], errors="coerce").astype(int)
    submission["opp_goals"] = pd.to_numeric(submission["opp_goals"], errors="coerce").astype(int)
    return submission

def _outcome(a: int, b: int) -> int:
    if a > b:
        return 1
    if a < b:
        return -1
    return 0

def get_tournament_weight(tournament: str) -> float:
    t = str(tournament).lower().strip()
    if "fifa world cup" in t or t == "world cup":
        return 2.00
    if "afc championship" in t or "afc asian cup" in t or "asian cup" in t:
        return 1.80
    if "friendly" in t:
        return 0.96
    return 1.20

def official_match_loss(y_team_true: int, y_opp_true: int, y_team_pred: int, y_opp_pred: int) -> float:
    y_team_true = int(y_team_true)
    y_opp_true = int(y_opp_true)
    y_team_pred = int(y_team_pred)
    y_opp_pred = int(y_opp_pred)

    mae = (abs(y_team_true - y_team_pred) + abs(y_opp_true - y_opp_pred)) / 2.0

    exact = int((y_team_true == y_team_pred) and (y_opp_true == y_opp_pred))
    outcome = int(_outcome(y_team_true, y_opp_true) == _outcome(y_team_pred, y_opp_pred))
    gd = int((y_team_true - y_opp_true) == (y_team_pred - y_opp_pred))

    penalty = (
        EXACT_PENALTY * (1 - exact) +
        OUTCOME_PENALTY * (1 - outcome) +
        GD_PENALTY * (1 - gd)
    )

    multiplier = 1.0 if outcome == 1 else WRONG_OUTCOME_MULTIPLIER
    raw_loss = mae + penalty
    return float((raw_loss * multiplier) ** NONLINEAR_POWER)

def awmae_score(y_team_true, y_opp_true, y_team_pred, y_opp_pred, tournaments) -> float:
    y_team_true = np.asarray(y_team_true)
    y_opp_true = np.asarray(y_opp_true)
    y_team_pred = np.asarray(y_team_pred)
    y_opp_pred = np.asarray(y_opp_pred)
    tournaments = np.asarray(tournaments)

    if not (len(y_team_true) == len(y_opp_true) == len(y_team_pred) == len(y_opp_pred) == len(tournaments)):
        raise ValueError("Panjang array input ke awmae_score tidak sama.")

    losses = np.array([
        official_match_loss(a, b, c, d)
        for a, b, c, d in zip(y_team_true, y_opp_true, y_team_pred, y_opp_pred)
    ], dtype=float)
    weights = np.array([get_tournament_weight(t) for t in tournaments], dtype=float)
    return float(np.sum(losses * weights) / np.sum(weights))

def make_time_based_holdout(train_match: pd.DataFrame, valid_fraction: float = 0.2):
    if "date" not in train_match.columns:
        raise KeyError("Kolom 'date' wajib ada untuk temporal holdout.")
    df = train_match.sort_values(["date", "match_id"]).reset_index(drop=True)
    n_total = len(df)
    n_valid = max(1, int(round(n_total * valid_fraction)))
    n_train = n_total - n_valid
    train_fold = df.iloc[:n_train].reset_index(drop=True)
    valid_fold = df.iloc[n_train:].reset_index(drop=True)

    print(
        f"Train fold: {len(train_fold):,} matches  "
        f"({train_fold['date'].min()} -> {train_fold['date'].max()})"
    )
    print(
        f"Valid fold: {len(valid_fold):,} matches  "
        f"({valid_fold['date'].min()} -> {valid_fold['date'].max()})"
    )
    return train_fold, valid_fold

def prepare_features(df: pd.DataFrame, cat_feats, num_feats) -> pd.DataFrame:
    """Siapkan DataFrame fitur untuk CatBoost.
    Categorical di-cast ke string; numeric dijaga agar tidak inf.
    """
    X = df[cat_feats + num_feats].copy()
    for col in cat_feats:
        X[col] = X[col].fillna("MISSING").astype(str)
    for col in num_feats:
        X[col] = pd.to_numeric(X[col], errors="coerce")
        X[col] = X[col].replace([np.inf, -np.inf], np.nan)
    return X

def build_outcome_target(goal_a: pd.Series, goal_b: pd.Series) -> pd.Series:
    goal_a = pd.to_numeric(goal_a, errors="coerce")
    goal_b = pd.to_numeric(goal_b, errors="coerce")
    out = np.where(goal_a > goal_b, 0, np.where(goal_a == goal_b, 1, 2))
    return pd.Series(out, index=goal_a.index, name="outcome_class")

def build_scoreline_prior(goal_a: pd.Series, goal_b: pd.Series, max_goals: int, alpha: float = 1.0) -> dict:
    counts = Counter()
    for a, b in zip(goal_a.astype(int), goal_b.astype(int)):
        a_clip = int(np.clip(a, 0, max_goals))
        b_clip = int(np.clip(b, 0, max_goals))
        counts[(a_clip, b_clip)] += 1

    prior = {}
    total = 0.0
    for a in range(max_goals + 1):
        for b in range(max_goals + 1):
            val = counts.get((a, b), 0) + alpha
            prior[(a, b)] = float(val)
            total += float(val)

    for k in prior:
        prior[k] /= total
    return prior

def decode_single_match_score(
    pred_goal_a: float,
    pred_goal_b: float,
    pred_total: float,
    pred_gd: float,
    pred_outcome_proba: np.ndarray,
    scoreline_prior: dict,
    max_goals: int,
    w_direct: float,
    w_total: float,
    w_gd: float,
    w_outcome: float,
    w_prior: float,
    eps: float = 1e-9,
):
    best_cost = np.inf
    best_pair = (0, 0)

    pred_goal_a = float(pred_goal_a)
    pred_goal_b = float(pred_goal_b)
    pred_total = float(pred_total)
    pred_gd = float(pred_gd)
    pred_outcome_proba = np.asarray(pred_outcome_proba, dtype=float).reshape(-1)

    for a in range(max_goals + 1):
        for b in range(max_goals + 1):
            outcome_idx = 0 if a > b else (1 if a == b else 2)

            cost = 0.0
            cost += w_direct * (abs(a - pred_goal_a) + abs(b - pred_goal_b))
            cost += w_total * abs((a + b) - pred_total)
            cost += w_gd * abs((a - b) - pred_gd)
            cost += w_outcome * (-math.log(float(pred_outcome_proba[outcome_idx]) + eps))
            cost += w_prior * (-math.log(float(scoreline_prior.get((a, b), eps)) + eps))

            if cost < best_cost:
                best_cost = cost
                best_pair = (int(a), int(b))

    return best_pair

def decode_batch_scores(
    pred_goal_a,
    pred_goal_b,
    pred_total,
    pred_gd,
    pred_outcome_proba,
    scoreline_prior: dict,
    max_goals: int,
    w_direct: float,
    w_total: float,
    w_gd: float,
    w_outcome: float,
    w_prior: float,
):
    decoded = [
        decode_single_match_score(
            pred_goal_a=float(ga),
            pred_goal_b=float(gb),
            pred_total=float(tg),
            pred_gd=float(gd),
            pred_outcome_proba=proba,
            scoreline_prior=scoreline_prior,
            max_goals=max_goals,
            w_direct=w_direct,
            w_total=w_total,
            w_gd=w_gd,
            w_outcome=w_outcome,
            w_prior=w_prior,
        )
        for ga, gb, tg, gd, proba in zip(
            pred_goal_a, pred_goal_b, pred_total, pred_gd, pred_outcome_proba
        )
    ]
    pred_a = np.array([x[0] for x in decoded], dtype=int)
    pred_b = np.array([x[1] for x in decoded], dtype=int)
    return pred_a, pred_b

def init_team_state() -> dict:
    return {
        "matches_played": 0,
        "last_match_date": pd.NaT,
        "elo_overall": DEFAULT_ELO,
        "elo_goal_diff": DEFAULT_ELO_GD,
        "ewm_points": DEFAULT_EWM_POINTS,
        "ewm_goals_for": DEFAULT_EWM_GF,
        "ewm_goals_against": DEFAULT_EWM_GA,
        "ewm_goal_diff": DEFAULT_EWM_GD,
        "points_last5": deque(maxlen=5),
        "points_last10": deque(maxlen=10),
        "gf_last5": deque(maxlen=5),
        "ga_last5": deque(maxlen=5),
        "gd_last5": deque(maxlen=5),
        "result_last10": deque(maxlen=10),
        "clean_sheet_last5": deque(maxlen=5),
        "failed_to_score_last5": deque(maxlen=5),
    }

def init_h2h_state() -> dict:
    return {
        "matches_played": 0,
        "points_a_last3": deque(maxlen=3),
        "gd_a_last3": deque(maxlen=3),
        "total_goals_last3": deque(maxlen=3),
    }

def snapshot_team_features(state: dict, match_date: pd.Timestamp) -> dict:
    last_date = state.get("last_match_date", pd.NaT)
    days_since = np.nan
    if pd.notna(last_date) and pd.notna(match_date):
        days_since = max((pd.Timestamp(match_date) - pd.Timestamp(last_date)).days, 0)

    result_last10 = list(state["result_last10"])
    win_rate_last10 = _safe_mean([1 if x == 1 else 0 for x in result_last10], default=np.nan)
    draw_rate_last10 = _safe_mean([1 if x == 0 else 0 for x in result_last10], default=np.nan)
    loss_rate_last10 = _safe_mean([1 if x == -1 else 0 for x in result_last10], default=np.nan)

    return {
        "hist_matches_played": float(state["matches_played"]),
        "hist_elo_overall": float(state["elo_overall"]),
        "hist_elo_gd": float(state["elo_goal_diff"]),
        "hist_ewm_points": float(state["ewm_points"]),
        "hist_ewm_gf": float(state["ewm_goals_for"]),
        "hist_ewm_ga": float(state["ewm_goals_against"]),
        "hist_ewm_gd": float(state["ewm_goal_diff"]),
        "hist_points_avg_last5": _safe_mean(state["points_last5"], default=np.nan),
        "hist_points_avg_last10": _safe_mean(state["points_last10"], default=np.nan),
        "hist_gf_avg_last5": _safe_mean(state["gf_last5"], default=np.nan),
        "hist_ga_avg_last5": _safe_mean(state["ga_last5"], default=np.nan),
        "hist_gd_avg_last5": _safe_mean(state["gd_last5"], default=np.nan),
        "hist_win_rate_last10": win_rate_last10,
        "hist_draw_rate_last10": draw_rate_last10,
        "hist_loss_rate_last10": loss_rate_last10,
        "hist_clean_sheet_rate_last5": _safe_mean(state["clean_sheet_last5"], default=np.nan),
        "hist_failed_to_score_rate_last5": _safe_mean(state["failed_to_score_last5"], default=np.nan),
        "hist_days_since_last_match": days_since,
        "hist_has_history": float(state["matches_played"] > 0),
    }

def snapshot_h2h_features(h2h_state: dict) -> dict:
    return {
        "h2h_matches_played_pre": float(h2h_state["matches_played"]),
        "h2h_points_a_avg_last3": _safe_mean(h2h_state["points_a_last3"], default=np.nan),
        "h2h_gd_a_avg_last3": _safe_mean(h2h_state["gd_a_last3"], default=np.nan),
        "h2h_total_goals_avg_last3": _safe_mean(h2h_state["total_goals_last3"], default=np.nan),
        "h2h_has_history": float(h2h_state["matches_played"] > 0),
    }

def expected_elo_result(rating_a: float, rating_b: float, home_bonus: float = 0.0) -> float:
    return 1.0 / (1.0 + 10.0 ** (-((rating_a + home_bonus) - rating_b) / 400.0))

def update_states_from_score(
    state_a: dict,
    state_b: dict,
    h2h_state: dict,
    match_context: dict,
    goals_a: int,
    goals_b: int,
):
    goals_a = int(goals_a)
    goals_b = int(goals_b)
    match_date = pd.Timestamp(match_context["date"])
    neutral = int(match_context.get("neutral", 0))
    team_a_is_home = int(match_context.get("team_a_is_home", 0))
    team_b_is_home = int(match_context.get("team_b_is_home", 0))
    tournament = match_context.get("tournament", "")

    if neutral == 1:
        elo_home_bonus = 0.0
        gd_home_bonus = 0.0
    else:
        if team_a_is_home == 1 and team_b_is_home != 1:
            elo_home_bonus = HOME_BONUS_ELO
            gd_home_bonus = HOME_BONUS_GD
        elif team_b_is_home == 1 and team_a_is_home != 1:
            elo_home_bonus = -HOME_BONUS_ELO
            gd_home_bonus = -HOME_BONUS_GD
        else:
            elo_home_bonus = 0.0
            gd_home_bonus = 0.0

    weight = get_tournament_weight(tournament)

    actual_a = 1.0 if goals_a > goals_b else (0.5 if goals_a == goals_b else 0.0)
    expected_a = expected_elo_result(state_a["elo_overall"], state_b["elo_overall"], elo_home_bonus)

    state_a["elo_overall"] += ELO_K * weight * (actual_a - expected_a)
    state_b["elo_overall"] -= ELO_K * weight * (actual_a - expected_a)

    expected_gd_a = ((state_a["elo_goal_diff"] + gd_home_bonus) - state_b["elo_goal_diff"]) / 100.0
    resid = (goals_a - goals_b) - expected_gd_a
    state_a["elo_goal_diff"] += GD_K * resid
    state_b["elo_goal_diff"] -= GD_K * resid

    points_a = _points_from_score(goals_a, goals_b)
    points_b = _points_from_score(goals_b, goals_a)
    gd_a = goals_a - goals_b
    gd_b = goals_b - goals_a

    # EWMA
    state_a["ewm_points"] = EWM_ALPHA * points_a + (1 - EWM_ALPHA) * state_a["ewm_points"]
    state_b["ewm_points"] = EWM_ALPHA * points_b + (1 - EWM_ALPHA) * state_b["ewm_points"]

    state_a["ewm_goals_for"] = EWM_ALPHA * goals_a + (1 - EWM_ALPHA) * state_a["ewm_goals_for"]
    state_b["ewm_goals_for"] = EWM_ALPHA * goals_b + (1 - EWM_ALPHA) * state_b["ewm_goals_for"]

    state_a["ewm_goals_against"] = EWM_ALPHA * goals_b + (1 - EWM_ALPHA) * state_a["ewm_goals_against"]
    state_b["ewm_goals_against"] = EWM_ALPHA * goals_a + (1 - EWM_ALPHA) * state_b["ewm_goals_against"]

    state_a["ewm_goal_diff"] = EWM_ALPHA * gd_a + (1 - EWM_ALPHA) * state_a["ewm_goal_diff"]
    state_b["ewm_goal_diff"] = EWM_ALPHA * gd_b + (1 - EWM_ALPHA) * state_b["ewm_goal_diff"]

    # Rolling deques
    state_a["points_last5"].append(points_a)
    state_b["points_last5"].append(points_b)
    state_a["points_last10"].append(points_a)
    state_b["points_last10"].append(points_b)

    state_a["gf_last5"].append(goals_a)
    state_b["gf_last5"].append(goals_b)
    state_a["ga_last5"].append(goals_b)
    state_b["ga_last5"].append(goals_a)

    state_a["gd_last5"].append(gd_a)
    state_b["gd_last5"].append(gd_b)

    state_a["result_last10"].append(_result_indicator(goals_a, goals_b))
    state_b["result_last10"].append(_result_indicator(goals_b, goals_a))

    state_a["clean_sheet_last5"].append(int(goals_b == 0))
    state_b["clean_sheet_last5"].append(int(goals_a == 0))
    state_a["failed_to_score_last5"].append(int(goals_a == 0))
    state_b["failed_to_score_last5"].append(int(goals_b == 0))

    state_a["matches_played"] += 1
    state_b["matches_played"] += 1
    state_a["last_match_date"] = match_date
    state_b["last_match_date"] = match_date

    # H2H state from perspective of team_a canonical
    h2h_state["matches_played"] += 1
    h2h_state["points_a_last3"].append(points_a)
    h2h_state["gd_a_last3"].append(gd_a)
    h2h_state["total_goals_last3"].append(goals_a + goals_b)

    return state_a, state_b, h2h_state

def build_history_feature_row(match_row: pd.Series, team_states: dict, h2h_states: dict) -> dict:
    gender = str(match_row["gender"])
    team_a = str(match_row["team_a"])
    team_b = str(match_row["team_b"])
    match_date = pd.Timestamp(match_row["date"])

    key_a = _team_state_key(gender, team_a)
    key_b = _team_state_key(gender, team_b)
    h2h_key = _canonical_pair_key(gender, team_a, team_b)

    state_a = team_states.get(key_a, init_team_state())
    state_b = team_states.get(key_b, init_team_state())
    h2h_state = h2h_states.get(h2h_key, init_h2h_state())

    feat_a = snapshot_team_features(state_a, match_date)
    feat_b = snapshot_team_features(state_b, match_date)
    feat_h2h = snapshot_h2h_features(h2h_state)

    row = {"match_id": match_row["match_id"]}
    row.update({f"{k}_a": v for k, v in feat_a.items()})
    row.update({f"{k}_b": v for k, v in feat_b.items()})
    row.update(feat_h2h)

    # matchup deltas
    diff_pairs = [
        ("hist_elo_overall", "hist_elo_overall_diff", "hist_elo_overall_abs_diff"),
        ("hist_elo_gd", "hist_elo_gd_diff", "hist_elo_gd_abs_diff"),
        ("hist_ewm_points", "hist_ewm_points_diff", None),
        ("hist_ewm_gf", "hist_ewm_gf_diff", None),
        ("hist_ewm_ga", "hist_ewm_ga_diff", None),
        ("hist_ewm_gd", "hist_ewm_gd_diff", None),
        ("hist_points_avg_last5", "hist_points_avg_last5_diff", None),
        ("hist_points_avg_last10", "hist_points_avg_last10_diff", None),
        ("hist_gf_avg_last5", "hist_gf_avg_last5_diff", None),
        ("hist_ga_avg_last5", "hist_ga_avg_last5_diff", None),
        ("hist_gd_avg_last5", "hist_gd_avg_last5_diff", None),
        ("hist_win_rate_last10", "hist_win_rate_last10_diff", None),
        ("hist_draw_rate_last10", "hist_draw_rate_last10_diff", None),
        ("hist_loss_rate_last10", "hist_loss_rate_last10_diff", None),
        ("hist_clean_sheet_rate_last5", "hist_clean_sheet_rate_last5_diff", None),
        ("hist_failed_to_score_rate_last5", "hist_failed_to_score_rate_last5_diff", None),
        ("hist_days_since_last_match", "hist_rest_days_diff", None),
        ("hist_matches_played", "hist_matches_played_diff", None),
    ]
    for base, diff_name, abs_name in diff_pairs:
        a_val = row.get(f"{base}_a", np.nan)
        b_val = row.get(f"{base}_b", np.nan)
        row[diff_name] = (a_val - b_val) if pd.notna(a_val) and pd.notna(b_val) else np.nan
        if abs_name is not None:
            row[abs_name] = abs(row[diff_name]) if pd.notna(row[diff_name]) else np.nan

    return row

def build_train_history_features(train_match_df: pd.DataFrame):
    team_states = {}
    h2h_states = {}
    rows = []

    df_sorted = train_match_df.sort_values(["date", "match_id"]).reset_index(drop=True)

    for match_row in tqdm(df_sorted.itertuples(index=False), total=len(df_sorted), desc="build_train_history_features"):
        row_dict = pd.Series(match_row._asdict())
        hist_row = build_history_feature_row(row_dict, team_states, h2h_states)
        rows.append(hist_row)

        gender = str(row_dict["gender"])
        team_a = str(row_dict["team_a"])
        team_b = str(row_dict["team_b"])
        key_a = _team_state_key(gender, team_a)
        key_b = _team_state_key(gender, team_b)
        h2h_key = _canonical_pair_key(gender, team_a, team_b)

        state_a = copy.deepcopy(team_states.get(key_a, init_team_state()))
        state_b = copy.deepcopy(team_states.get(key_b, init_team_state()))
        h2h_state = copy.deepcopy(h2h_states.get(h2h_key, init_h2h_state()))

        match_context = {
            "date": row_dict["date"],
            "neutral": row_dict.get("neutral", 0),
            "team_a_is_home": row_dict.get("team_a_is_home", 0),
            "team_b_is_home": row_dict.get("team_b_is_home", 0),
            "tournament": row_dict.get("tournament", ""),
        }
        state_a, state_b, h2h_state = update_states_from_score(
            state_a, state_b, h2h_state,
            match_context=match_context,
            goals_a=int(row_dict["team_a_goals"]),
            goals_b=int(row_dict["team_b_goals"]),
        )

        team_states[key_a] = state_a
        team_states[key_b] = state_b
        h2h_states[h2h_key] = h2h_state

    hist_df = pd.DataFrame(rows)
    return hist_df, team_states, h2h_states

def team_states_to_frame(team_states: dict) -> pd.DataFrame:
    rows = []
    for (gender, team), state in team_states.items():
        snap = snapshot_team_features(state, match_date=state.get("last_match_date", pd.NaT))
        row = {"gender": gender, "team": team}
        row.update(snap)
        row["last_match_date"] = state.get("last_match_date", pd.NaT)
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["gender", "team"]).reset_index(drop=True)

print("[OK] Semua helper inti berhasil didefinisikan.")


[OK] Semua helper inti berhasil didefinisikan.


### Catatan Helper

Notebook ini sengaja membawa ulang helper dari fondasi EXP 00/01, lalu menambahkan helper khusus historical strength engine.  
Targetnya sederhana: **self-contained, mudah diaudit, dan tidak bergantung pada state notebook lain**.


## 04. Cleaning Awal dan Canonical Base Match Data


In [18]:
EXPECTED_SHARED_COLS = [
    "Id", "match_id", "date", "gender", "team", "opponent",
    "is_home", "neutral", "tournament", "venue_country",
    "confederation_team", "confederation_opp",
    "population_team", "population_opp",
    "gdp_per_capita_team", "gdp_per_capita_opp",
    "altitude_venue", "distance_travel_team", "distance_travel_opp",
    "temperature_venue",
]
TARGET_COLS = ["team_goals", "opp_goals"]

for col in EXPECTED_SHARED_COLS:
    assert col in train.columns, f"{col} tidak ada di train"
    assert col in test.columns, f"{col} tidak ada di test"
for col in TARGET_COLS:
    assert col in train.columns, f"{col} tidak ada di train"

print("[OK] Shared columns dan target minimum tersedia.")


[OK] Shared columns dan target minimum tersedia.


In [19]:
# Cleaning konservatif
for name, df in [("train", train), ("test", test)]:
    if "altitude_venue" in df.columns:
        n_sentinel = int((df["altitude_venue"] == -9999).sum())
        df.loc[df["altitude_venue"] == -9999, "altitude_venue"] = np.nan
        print(f"  {name}: {n_sentinel:,} sentinel altitude_venue -> NaN")

string_cols = [
    "team", "opponent", "gender", "tournament", "venue_country",
    "confederation_team", "confederation_opp",
]
for df in [train, test]:
    for col in string_cols:
        df[col] = df[col].fillna("UNKNOWN").astype(str)

train["date"] = pd.to_datetime(train["date"], errors="coerce")
test["date"] = pd.to_datetime(test["date"], errors="coerce")

if train["date"].isna().any() or test["date"].isna().any():
    raise ValueError("Ada date yang gagal diparse menjadi datetime.")

print("[OK] Cleaning awal selesai.")


  train: 766 sentinel altitude_venue -> NaN
  test: 296 sentinel altitude_venue -> NaN
[OK] Cleaning awal selesai.


In [20]:
train_match_base = build_match_level(train, is_train=True)
test_match_base = build_match_level(test, is_train=False)

print(f"train_match_base: {train_match_base.shape}")
print(f"test_match_base : {test_match_base.shape}")
print("[OK] Canonical base match data berhasil dibangun.")


train_match_base: (39386, 24)
test_match_base : (21211, 22)
[OK] Canonical base match data berhasil dibangun.


In [21]:
display(train_match_base.head(3))
display(test_match_base.head(3))

,match_id,date,gender,tournament,venue_country,neutral,altitude_venue,temperature_venue,team_a,team_b,team_a_is_home,team_b_is_home,team_a_confederation,team_b_confederation,team_a_population,team_b_population,team_a_gdp_per_capita,team_b_gdp_per_capita,team_a_distance_travel,team_b_distance_travel,row_id_a,row_id_b,team_a_goals,team_b_goals
0,M000001,1872-11-30,M,Friendly,Scotland,0,NaN,NaN,England,Scotland,0.0000,1.0000,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,M000001_England,M000001_Scotland,0.0000,0.0000
1,M000002,1873-03-08,M,Friendly,England,0,NaN,NaN,England,Scotland,1.0000,0.0000,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,M000002_England,M000002_Scotland,4.0000,2.0000
2,M000003,1874-03-07,M,Friendly,Scotland,0,NaN,NaN,England,Scotland,0.0000,1.0000,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,M000003_England,M000003_Scotland,1.0000,2.0000


,match_id,date,gender,tournament,venue_country,neutral,altitude_venue,temperature_venue,team_a,team_b,team_a_is_home,team_b_is_home,team_a_confederation,team_b_confederation,team_a_population,team_b_population,team_a_gdp_per_capita,team_b_gdp_per_capita,team_a_distance_travel,team_b_distance_travel,row_id_a,row_id_b
0,M034984,2011-08-06,M,Indian Ocean Island Games,Seychelles,0,NaN,25.7902,Mauritius,Seychelles,0.0000,1.0000,CAF,CAF,1283330.0000,92409.0000,9197.0270,12189.0952,1751.8957,0.0000,M034984_Mauritius,M034984_Seychelles
1,M034985,2011-08-06,M,Indian Ocean Island Games,Seychelles,1,NaN,25.7902,Comoros,Maldives,1.0000,0.0000,CAF,AFC,656024.0000,361575.0000,1447.9451,7291.4660,1517.0107,5483.1175,M034985_Comoros,M034985_Maldives
2,M034986,2011-08-06,M,Indian Ocean Island Games,Seychelles,1,NaN,25.7902,Madagascar,Réunion,0.0000,1.0000,CAF,Unknown,21731053.0000,NaN,531.2654,NaN,NaN,NaN,M034986_Madagascar,M034986_Réunion


Canonicalization **tetap sama** dengan eksperimen sebelumnya, yaitu urut alfabetis untuk menentukan `team_a` dan `team_b`.  
Ini penting agar perbandingan antar eksperimen tetap adil dan tidak tercampur oleh perubahan definisi sisi A/B.


## 05. Static Shared Features ala EXP 01


In [22]:
def engineer_static_match_features(df_match: pd.DataFrame) -> pd.DataFrame:
    df = df_match.copy()

    dt = pd.to_datetime(df["date"], errors="coerce")
    df["match_year"] = dt.dt.year
    df["match_month"] = dt.dt.month
    df["match_quarter"] = dt.dt.quarter
    df["match_dayofweek"] = dt.dt.dayofweek
    df["match_dayofyear"] = dt.dt.dayofyear
    df["match_is_weekend"] = (dt.dt.dayofweek >= 5).astype(int)
    df["match_decade"] = (dt.dt.year // 10) * 10

    df["home_side"] = np.where(
        pd.to_numeric(df["team_a_is_home"], errors="coerce").fillna(0) == 1, 1,
        np.where(pd.to_numeric(df["team_b_is_home"], errors="coerce").fillna(0) == 1, -1, 0)
    )

    conf_a = df["team_a_confederation"].fillna("UNKNOWN").astype(str)
    conf_b = df["team_b_confederation"].fillna("UNKNOWN").astype(str)
    df["same_confederation"] = (conf_a == conf_b).astype(int)

    t_low = df["tournament"].fillna("").astype(str).str.lower()
    df["is_friendly"] = t_low.str.contains("friendly").astype(int)
    df["is_world_cup"] = (t_low.eq("world cup") | t_low.str.contains("fifa world cup")).astype(int)
    df["is_qualification"] = t_low.str.contains("qualif").astype(int)
    df["is_nations_league"] = t_low.str.contains("nations league").astype(int)
    df["tournament_weight_proxy"] = df["tournament"].apply(get_tournament_weight).astype(float)

    numeric_pairs = [
        ("population", "team_a_population", "team_b_population"),
        ("gdp_per_capita", "team_a_gdp_per_capita", "team_b_gdp_per_capita"),
        ("distance_travel", "team_a_distance_travel", "team_b_distance_travel"),
    ]
    for name, col_a, col_b in numeric_pairs:
        df[col_a] = pd.to_numeric(df[col_a], errors="coerce")
        df[col_b] = pd.to_numeric(df[col_b], errors="coerce")

        df[f"{name}_diff"] = df[col_a] - df[col_b]
        df[f"{name}_abs_diff"] = (df[col_a] - df[col_b]).abs()
        df[f"log_{name}_a"] = _safe_log1p(df[col_a])
        df[f"log_{name}_b"] = _safe_log1p(df[col_b])
        df[f"log_{name}_diff"] = df[f"log_{name}_a"] - df[f"log_{name}_b"]
        df[f"{name}_ratio_ab"] = _safe_divide(df[col_a], df[col_b])

    for col in ["altitude_venue", "temperature_venue"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[col] = df[col].replace([np.inf, -np.inf], np.nan)

    df["pair_key"] = df["team_a"].astype(str) + "__VS__" + df["team_b"].astype(str)
    df["confed_pair_key"] = (
        df["team_a_confederation"].fillna("UNKNOWN").astype(str)
        + "__VS__" +
        df["team_b_confederation"].fillna("UNKNOWN").astype(str)
    )

    return df

train_match_static = engineer_static_match_features(train_match_base)
test_match_static = engineer_static_match_features(test_match_base)

print(f"train_match_static: {train_match_static.shape}")
print(f"test_match_static : {test_match_static.shape}")
print("[OK] Static feature block ala EXP 01 selesai dibangun.")


train_match_static: (39386, 58)
test_match_static : (21211, 56)
[OK] Static feature block ala EXP 01 selesai dibangun.


Static shared feature block ini sengaja dipertahankan agar EXP 02 punya **control baseline** yang fair.  
Jadi, tambahan value utama di eksperimen ini benar-benar datang dari **historical strength features**, bukan dari pergantian model dasar.


## 06. Historical Strength Engine — Definisi State dan Helper


In [23]:
# Quick sanity check untuk helper history
tmp_state = init_team_state()
tmp_h2h = init_h2h_state()
tmp_snap = snapshot_team_features(tmp_state, pd.Timestamp("2000-01-01"))
tmp_h2h_snap = snapshot_h2h_features(tmp_h2h)

print("[INFO] Snapshot team state default:")
display(pd.DataFrame([tmp_snap]))
print("[INFO] Snapshot H2H state default:")
display(pd.DataFrame([tmp_h2h_snap]))


[INFO] Snapshot team state default:


,hist_matches_played,hist_elo_overall,hist_elo_gd,hist_ewm_points,hist_ewm_gf,hist_ewm_ga,hist_ewm_gd,hist_points_avg_last5,hist_points_avg_last10,hist_gf_avg_last5,hist_ga_avg_last5,hist_gd_avg_last5,hist_win_rate_last10,hist_draw_rate_last10,hist_loss_rate_last10,hist_clean_sheet_rate_last5,hist_failed_to_score_rate_last5,hist_days_since_last_match,hist_has_history
0,0.0000,1500.0000,0.0000,1.0000,1.2000,1.2000,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000


[INFO] Snapshot H2H state default:


,h2h_matches_played_pre,h2h_points_a_avg_last3,h2h_gd_a_avg_last3,h2h_total_goals_avg_last3,h2h_has_history
0,0.0000,NaN,NaN,NaN,0.0000


### Catatan Historical Engine

State disimpan per key **`(gender, team)`**, bukan hanya per tim mentah.  
Ini sengaja dilakukan karena distribusi men’s dan women’s match tidak identik, dan EXP 01 sudah memberi sinyal bahwa keduanya memang berbeda.

Selain team-level state, kita juga menyimpan lightweight H2H state per canonical pair **`(gender, team_a, team_b)`**.  
Tujuannya bukan membuat engine yang terlalu kompleks, tetapi memberi sedikit konteks matchup historis yang tetap leakage-safe.


## 07. Build Leakage-Safe Historical Features untuk Train


In [24]:
# Build history features full-train (actual pre-match)
train_history_full_df, final_team_states_full_train, final_h2h_states_full_train = build_train_history_features(train_match_base)

train_match_hist_full = train_match_base.merge(
    train_history_full_df,
    on="match_id",
    how="left",
    validate="one_to_one"
)

print(f"train_history_full_df : {train_history_full_df.shape}")
print(f"train_match_hist_full : {train_match_hist_full.shape}")
print("[OK] Historical pre-match features full-train berhasil dibangun.")


build_train_history_features:   0%|          | 0/39386 [00:00<?, ?it/s]

train_history_full_df : (39386, 64)
train_match_hist_full : (39386, 87)
[OK] Historical pre-match features full-train berhasil dibangun.


In [25]:
cutoff_team_states_preview = team_states_to_frame(final_team_states_full_train)
cutoff_team_states_preview.to_csv(f"{SUM_DIR}/cutoff_team_states.csv", index=False)

print(f"[OK] Snapshot final full-train team states disimpan ke {SUM_DIR}/cutoff_team_states.csv")
display(cutoff_team_states_preview.head(10))


[OK] Snapshot final full-train team states disimpan ke ../outputs/exp02_history_strength_engine_recursive/summaries/cutoff_team_states.csv


,gender,team,hist_matches_played,hist_elo_overall,hist_elo_gd,hist_ewm_points,hist_ewm_gf,hist_ewm_ga,hist_ewm_gd,hist_points_avg_last5,hist_points_avg_last10,hist_gf_avg_last5,hist_ga_avg_last5,hist_gd_avg_last5,hist_win_rate_last10,hist_draw_rate_last10,hist_loss_rate_last10,hist_clean_sheet_rate_last5,hist_failed_to_score_rate_last5,hist_days_since_last_match,hist_has_history,last_match_date
0,M,Afghanistan,56.0000,1192.6813,-275.2029,0.8437,0.6219,1.2885,-0.6666,0.8000,1.0000,0.4000,1.2000,-0.8000,0.3000,0.1000,0.6000,0.2000,0.6000,0,1.0000,2011-07-03
1,M,Albania,249.0000,1521.4818,71.9878,0.6494,0.3197,2.1712,-1.8515,0.8000,1.2000,0.4000,1.6000,-1.2000,0.3000,0.3000,0.4000,0.4000,0.6000,0,1.0000,2011-06-20
2,M,Alderney,108.0000,1144.7184,-342.8528,0.0060,0.1121,4.1631,-4.0510,0.0000,0.0000,0.2000,4.6000,-4.4000,0.0000,0.0000,1.0000,0.0000,0.8000,0,1.0000,2011-06-30
3,M,Algeria,438.0000,1581.9926,36.3978,0.9369,0.3412,1.8066,-1.4654,1.0000,0.9000,0.4000,1.4000,-1.0000,0.2000,0.3000,0.5000,0.4000,0.6000,0,1.0000,2011-06-04
4,M,Ambazonia,6.0000,1434.3284,-72.0616,0.1160,0.5217,2.8236,-2.3019,0.0000,0.1667,0.2000,3.0000,-2.8000,0.0000,0.1667,0.8333,0.0000,0.8000,0,1.0000,2007-06-07
5,M,American Samoa,31.0000,1200.1902,-705.1044,0.0000,0.1327,8.6251,-8.4924,0.0000,0.0000,0.2000,9.6000,-9.4000,0.0000,0.0000,1.0000,0.0000,0.8000,0,1.0000,2007-09-01
6,M,Andalusia,12.0000,1572.3481,98.1547,2.5785,2.8546,0.8745,1.9802,2.6000,2.4000,2.4000,0.8000,1.6000,0.7000,0.3000,0.0000,0.4000,0.2000,0,1.0000,2007-12-27
7,M,Andorra,98.0000,1151.3245,-122.0566,0.0000,0.1929,1.6782,-1.4853,0.0000,0.0000,0.2000,2.0000,-1.8000,0.0000,0.0000,1.0000,0.0000,0.8000,0,1.0000,2011-06-04
8,M,Angola,269.0000,1531.3373,58.7058,1.5266,0.7608,0.6950,0.0658,1.4000,1.1000,0.8000,0.6000,0.2000,0.3000,0.2000,0.5000,0.6000,0.4000,0,1.0000,2011-06-05
9,M,Anguilla,43.0000,1177.2613,-388.2434,0.4370,0.3599,2.5931,-2.2332,0.8000,0.4000,0.6000,2.2000,-1.6000,0.1000,0.1000,0.8000,0.2000,0.6000,0,1.0000,2011-07-10


Urutan proses history di train **selalu**:

1. snapshot fitur **pre-match**,
2. simpan feature row,
3. baru update state dengan hasil aktual match.

Dengan urutan ini, setiap baris train benar-benar hanya melihat histori sebelum pertandingan tersebut.


## 08. Temporal Holdout dan State Cutoff


In [26]:
train_fold_base, valid_fold_base = make_time_based_holdout(train_match_base, valid_fraction=0.2)

train_overlap = set(train_fold_base["match_id"]) & set(valid_fold_base["match_id"])
assert len(train_overlap) == 0, "Ada overlap match_id antara train_fold dan valid_fold"

print(f"[OK] Tidak ada overlap match_id antara train_fold dan valid_fold.")


Train fold: 31,509 matches  (1872-11-30 00:00:00 -> 2005-02-01 00:00:00)
Valid fold: 7,877 matches  (2005-02-01 00:00:00 -> 2011-08-04 00:00:00)
[OK] Tidak ada overlap match_id antara train_fold dan valid_fold.


In [27]:
# Build actual train history features hanya dari train_fold
train_fold_history_df, cutoff_team_states_valid, cutoff_h2h_states_valid = build_train_history_features(train_fold_base)

train_fold_static = engineer_static_match_features(train_fold_base)
valid_fold_static = engineer_static_match_features(valid_fold_base)

train_fold_hist = train_fold_base.merge(train_fold_history_df, on="match_id", how="left", validate="one_to_one")

print(f"train_fold_history_df: {train_fold_history_df.shape}")
print(f"train_fold_hist      : {train_fold_hist.shape}")
print("[OK] Cutoff state untuk validation berhasil dibangun dari train_fold saja.")


build_train_history_features:   0%|          | 0/31509 [00:00<?, ?it/s]

train_fold_history_df: (31509, 64)
train_fold_hist      : (31509, 87)
[OK] Cutoff state untuk validation berhasil dibangun dari train_fold saja.


Validation di notebook ini diperlakukan sebagai **future simulation**.  
Artinya state awal validation hanya boleh berasal dari histori sampai cutoff akhir `train_fold`, dan **tidak boleh** memakai actual outcome dari `valid_fold` untuk memperkaya state inferensi.


## 09. Target Construction dan Final Feature Set


In [28]:
# Static feature blocks
static_categorical_features = [
    "team_a", "team_b",
    "gender", "tournament", "venue_country",
    "team_a_confederation", "team_b_confederation",
    "pair_key", "confed_pair_key",
]

static_numeric_features = [
    "match_year", "match_month", "match_quarter",
    "match_dayofweek", "match_dayofyear", "match_is_weekend", "match_decade",
    "neutral", "team_a_is_home", "team_b_is_home", "home_side",
    "same_confederation",
    "is_friendly", "is_world_cup", "is_qualification", "is_nations_league",
    "tournament_weight_proxy",
    "altitude_venue", "temperature_venue",
    "team_a_population", "team_b_population",
    "population_diff", "population_abs_diff",
    "log_population_a", "log_population_b", "log_population_diff",
    "population_ratio_ab",
    "team_a_gdp_per_capita", "team_b_gdp_per_capita",
    "gdp_per_capita_diff", "gdp_per_capita_abs_diff",
    "log_gdp_per_capita_a", "log_gdp_per_capita_b", "log_gdp_per_capita_diff",
    "gdp_per_capita_ratio_ab",
    "team_a_distance_travel", "team_b_distance_travel",
    "distance_travel_diff", "distance_travel_abs_diff",
    "log_distance_travel_a", "log_distance_travel_b", "log_distance_travel_diff",
    "distance_travel_ratio_ab",
]

history_numeric_features = [
    "hist_matches_played_a", "hist_matches_played_b",
    "hist_elo_overall_a", "hist_elo_overall_b",
    "hist_elo_gd_a", "hist_elo_gd_b",
    "hist_ewm_points_a", "hist_ewm_points_b",
    "hist_ewm_gf_a", "hist_ewm_gf_b",
    "hist_ewm_ga_a", "hist_ewm_ga_b",
    "hist_ewm_gd_a", "hist_ewm_gd_b",
    "hist_points_avg_last5_a", "hist_points_avg_last5_b",
    "hist_points_avg_last10_a", "hist_points_avg_last10_b",
    "hist_gf_avg_last5_a", "hist_gf_avg_last5_b",
    "hist_ga_avg_last5_a", "hist_ga_avg_last5_b",
    "hist_gd_avg_last5_a", "hist_gd_avg_last5_b",
    "hist_win_rate_last10_a", "hist_win_rate_last10_b",
    "hist_draw_rate_last10_a", "hist_draw_rate_last10_b",
    "hist_loss_rate_last10_a", "hist_loss_rate_last10_b",
    "hist_clean_sheet_rate_last5_a", "hist_clean_sheet_rate_last5_b",
    "hist_failed_to_score_rate_last5_a", "hist_failed_to_score_rate_last5_b",
    "hist_days_since_last_match_a", "hist_days_since_last_match_b",
    "hist_has_history_a", "hist_has_history_b",
    "hist_elo_overall_diff", "hist_elo_overall_abs_diff",
    "hist_elo_gd_diff", "hist_elo_gd_abs_diff",
    "hist_ewm_points_diff", "hist_ewm_gf_diff", "hist_ewm_ga_diff", "hist_ewm_gd_diff",
    "hist_points_avg_last5_diff", "hist_points_avg_last10_diff",
    "hist_gf_avg_last5_diff", "hist_ga_avg_last5_diff", "hist_gd_avg_last5_diff",
    "hist_win_rate_last10_diff", "hist_draw_rate_last10_diff", "hist_loss_rate_last10_diff",
    "hist_clean_sheet_rate_last5_diff", "hist_failed_to_score_rate_last5_diff",
    "hist_rest_days_diff", "hist_matches_played_diff",
    "h2h_matches_played_pre", "h2h_points_a_avg_last3",
    "h2h_gd_a_avg_last3", "h2h_total_goals_avg_last3", "h2h_has_history",
]

control_features = static_categorical_features + static_numeric_features
history_features = static_categorical_features + static_numeric_features + history_numeric_features

target_info = {
    "y_goal_a": "team_a_goals",
    "y_goal_b": "team_b_goals",
    "y_total_goals": "team_a_goals + team_b_goals",
    "y_goal_diff": "team_a_goals - team_b_goals",
    "y_outcome_class": "{0: team_a_win, 1: draw, 2: team_b_win}",
}

feature_info = {
    "static_categorical_features": static_categorical_features,
    "static_numeric_features": static_numeric_features,
    "history_numeric_features": history_numeric_features,
    "control_features": control_features,
    "history_features": history_features,
}

with open(f"{SUM_DIR}/feature_columns.json", "w") as f:
    json.dump({
        "static_categorical_features": static_categorical_features,
        "static_numeric_features": static_numeric_features,
        "control_features_count": len(control_features),
        "history_features_count": len(history_features),
    }, f, indent=2)

with open(f"{SUM_DIR}/history_feature_columns.json", "w") as f:
    json.dump({
        "history_numeric_features": history_numeric_features,
        "history_numeric_features_count": len(history_numeric_features),
    }, f, indent=2)

print(f"[OK] Feature columns disimpan ke {SUM_DIR}/feature_columns.json")
print(f"[OK] History feature columns disimpan ke {SUM_DIR}/history_feature_columns.json")
print(f"Static categorical: {len(static_categorical_features)}")
print(f"Static numeric    : {len(static_numeric_features)}")
print(f"History numeric   : {len(history_numeric_features)}")
print(f"Control features  : {len(control_features)}")
print(f"History features  : {len(history_features)}")


[OK] Feature columns disimpan ke ../outputs/exp02_history_strength_engine_recursive/summaries/feature_columns.json
[OK] History feature columns disimpan ke ../outputs/exp02_history_strength_engine_recursive/summaries/history_feature_columns.json
Static categorical: 9
Static numeric    : 43
History numeric   : 63
Control features  : 52
History features  : 115


Feature set di EXP 02 dibagi jelas menjadi tiga blok:

1. **static categorical**,
2. **static numeric**,
3. **history numeric**.

Dengan pemisahan ini, perbandingan control vs history-augmented bisa dilakukan dengan lebih fair dan lebih mudah diaudit.


## 10. Baseline Control — Reproduce EXP 01 Core


In [29]:
def train_catboost_decomposition_models(
    train_df: pd.DataFrame,
    categorical_features: list,
    numeric_features: list,
    label_prefix: str = "model",
):
    feature_cols = categorical_features + numeric_features
    X_train = prepare_features(train_df, categorical_features, numeric_features)

    y_goal_a = train_df["team_a_goals"].astype(float).values
    y_goal_b = train_df["team_b_goals"].astype(float).values
    y_total = (train_df["team_a_goals"] + train_df["team_b_goals"]).astype(float).values
    y_gd = (train_df["team_a_goals"] - train_df["team_b_goals"]).astype(float).values
    y_outcome = build_outcome_target(train_df["team_a_goals"], train_df["team_b_goals"]).astype(int).values

    goal_reg_params = dict(
        loss_function="RMSE",
        eval_metric="RMSE",
        iterations=1500,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=5.0,
        random_seed=SEED,
        allow_writing_files=False,
        verbose=200,
    )
    outcome_clf_params = dict(
        loss_function="MultiClass",
        eval_metric="MultiClass",
        iterations=1500,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=5.0,
        random_seed=SEED,
        allow_writing_files=False,
        verbose=200,
    )

    print(f"[INFO] Training {label_prefix}_goal_a ...")
    model_goal_a = CatBoostRegressor(**goal_reg_params)
    model_goal_a.fit(Pool(X_train, y_goal_a, cat_features=categorical_features), verbose=200)

    print(f"[INFO] Training {label_prefix}_goal_b ...")
    model_goal_b = CatBoostRegressor(**goal_reg_params)
    model_goal_b.fit(Pool(X_train, y_goal_b, cat_features=categorical_features), verbose=200)

    print(f"[INFO] Training {label_prefix}_total_goals ...")
    model_total = CatBoostRegressor(**goal_reg_params)
    model_total.fit(Pool(X_train, y_total, cat_features=categorical_features), verbose=200)

    print(f"[INFO] Training {label_prefix}_goal_diff ...")
    model_gd = CatBoostRegressor(**goal_reg_params)
    model_gd.fit(Pool(X_train, y_gd, cat_features=categorical_features), verbose=200)

    print(f"[INFO] Training {label_prefix}_outcome ...")
    model_outcome = CatBoostClassifier(**outcome_clf_params)
    model_outcome.fit(Pool(X_train, y_outcome, cat_features=categorical_features), verbose=200)

    return {
        "model_goal_a": model_goal_a,
        "model_goal_b": model_goal_b,
        "model_total_goals": model_total,
        "model_goal_diff": model_gd,
        "model_outcome": model_outcome,
        "categorical_features": categorical_features,
        "numeric_features": numeric_features,
        "feature_cols": feature_cols,
    }

def predict_decomposition_outputs(
    df_features: pd.DataFrame,
    trained_models: dict,
):
    cat_feats = trained_models["categorical_features"]
    num_feats = trained_models["numeric_features"]
    X = prepare_features(df_features, cat_feats, num_feats)

    pred_goal_a = trained_models["model_goal_a"].predict(X)
    pred_goal_b = trained_models["model_goal_b"].predict(X)
    pred_total = trained_models["model_total_goals"].predict(X)
    pred_gd = trained_models["model_goal_diff"].predict(X)
    pred_outcome_proba = trained_models["model_outcome"].predict_proba(X)

    return {
        "pred_goal_a_cont": np.asarray(pred_goal_a, dtype=float),
        "pred_goal_b_cont": np.asarray(pred_goal_b, dtype=float),
        "pred_total_cont": np.asarray(pred_total, dtype=float),
        "pred_gd_cont": np.asarray(pred_gd, dtype=float),
        "pred_outcome_proba": np.asarray(pred_outcome_proba, dtype=float),
    }

def tune_decoder_from_raw_outputs(
    raw_outputs: dict,
    tournaments,
    y_true_a,
    y_true_b,
    scoreline_prior: dict,
    variant_name: str,
    decoder_grid: dict,
):
    rows = []
    best_score = np.inf
    best_params = None
    best_preds = None

    combos = list(product(
        decoder_grid["MAX_GOALS"],
        decoder_grid["w_direct"],
        decoder_grid["w_total"],
        decoder_grid["w_gd"],
        decoder_grid["w_outcome"],
        decoder_grid["w_prior"],
    ))

    for max_goals, w_direct, w_total, w_gd, w_outcome, w_prior in tqdm(combos, desc=f"tune_decoder_{variant_name}"):
        pred_a, pred_b = decode_batch_scores(
            pred_goal_a=raw_outputs["pred_goal_a_cont"],
            pred_goal_b=raw_outputs["pred_goal_b_cont"],
            pred_total=raw_outputs["pred_total_cont"],
            pred_gd=raw_outputs["pred_gd_cont"],
            pred_outcome_proba=raw_outputs["pred_outcome_proba"],
            scoreline_prior=scoreline_prior,
            max_goals=max_goals,
            w_direct=w_direct,
            w_total=w_total,
            w_gd=w_gd,
            w_outcome=w_outcome,
            w_prior=w_prior,
        )
        score = awmae_score(y_true_a, y_true_b, pred_a, pred_b, tournaments)
        row = {
            "variant_name": variant_name,
            "MAX_GOALS": max_goals,
            "w_direct": w_direct,
            "w_total": w_total,
            "w_gd": w_gd,
            "w_outcome": w_outcome,
            "w_prior": w_prior,
            "valid_awmae": score,
        }
        rows.append(row)

        if score < best_score:
            best_score = score
            best_params = row.copy()
            best_preds = (pred_a.copy(), pred_b.copy())

    grid_df = pd.DataFrame(rows).sort_values("valid_awmae").reset_index(drop=True)
    return grid_df, best_params, best_preds

def build_prediction_frame(
    df_base: pd.DataFrame,
    pred_a: np.ndarray,
    pred_b: np.ndarray,
    raw_outputs=None,
    variant_name: str = "variant",
) -> pd.DataFrame:
    out = df_base[["match_id", "date", "gender", "tournament", "neutral", "team_a", "team_b"]].copy()
    out["pred_team_a_goals"] = pred_a.astype(int)
    out["pred_team_b_goals"] = pred_b.astype(int)
    out["variant_name"] = variant_name

    if "team_a_goals" in df_base.columns:
        out["actual_team_a_goals"] = df_base["team_a_goals"].astype(int).values
        out["actual_team_b_goals"] = df_base["team_b_goals"].astype(int).values
        out["awmae_match_loss"] = [
            official_match_loss(ta, tb, pa, pb)
            for ta, tb, pa, pb in zip(
                out["actual_team_a_goals"],
                out["actual_team_b_goals"],
                out["pred_team_a_goals"],
                out["pred_team_b_goals"],
            )
        ]

    if raw_outputs is not None:
        for key, arr in raw_outputs.items():
            if key == "pred_outcome_proba":
                out["pred_outcome_team_a_win"] = arr[:, 0]
                out["pred_outcome_draw"] = arr[:, 1]
                out["pred_outcome_team_b_win"] = arr[:, 2]
            else:
                out[key] = np.asarray(arr)

    return out

print("[OK] Helper training, prediction, dan decoder tuning siap.")


[OK] Helper training, prediction, dan decoder tuning siap.


In [30]:
# Control training data = static only
control_train_df = train_fold_static.copy()
control_valid_df = valid_fold_static.copy()

control_models = train_catboost_decomposition_models(
    train_df=control_train_df,
    categorical_features=static_categorical_features,
    numeric_features=static_numeric_features,
    label_prefix="control",
)


[INFO] Training control_goal_a ...
0:	learn: 1.7984995	total: 274ms	remaining: 6m 50s
200:	learn: 1.4956637	total: 23.1s	remaining: 2m 29s
400:	learn: 1.4389065	total: 47.6s	remaining: 2m 10s
600:	learn: 1.3953698	total: 1m 12s	remaining: 1m 47s
800:	learn: 1.3597319	total: 1m 36s	remaining: 1m 24s
1000:	learn: 1.3304272	total: 2m	remaining: 59.9s
1200:	learn: 1.3023928	total: 2m 24s	remaining: 36s
1400:	learn: 1.2795854	total: 2m 48s	remaining: 11.9s
1499:	learn: 1.2692060	total: 3m 1s	remaining: 0us
[INFO] Training control_goal_b ...
0:	learn: 1.7798285	total: 110ms	remaining: 2m 44s
200:	learn: 1.4879281	total: 23.3s	remaining: 2m 30s
400:	learn: 1.4360678	total: 46.5s	remaining: 2m 7s
600:	learn: 1.3980686	total: 1m 10s	remaining: 1m 45s
800:	learn: 1.3694094	total: 1m 34s	remaining: 1m 22s
1000:	learn: 1.3404756	total: 1m 58s	remaining: 59.3s
1200:	learn: 1.3136650	total: 2m 23s	remaining: 35.8s
1400:	learn: 1.2892914	total: 2m 48s	remaining: 11.9s
1499:	learn: 1.2767275	total: 3m

In [31]:
# Raw control predictions on valid
control_raw_valid = predict_decomposition_outputs(control_valid_df, control_models)

DECODER_GRID = {
    "MAX_GOALS": [6, 7, 8],
    "w_direct": [0.5, 1.0],
    "w_total": [1.0, 1.5],
    "w_gd": [1.0, 1.5],
    "w_outcome": [1.0, 2.0, 2.5],
    "w_prior": [0.2, 0.5],
}

control_scoreline_prior = build_scoreline_prior(
    train_fold_base["team_a_goals"].astype(int),
    train_fold_base["team_b_goals"].astype(int),
    max_goals=max(DECODER_GRID["MAX_GOALS"]),
    alpha=1.0,
)

decoder_grid_control_df, best_control_params, best_control_preds = tune_decoder_from_raw_outputs(
    raw_outputs=control_raw_valid,
    tournaments=valid_fold_base["tournament"],
    y_true_a=valid_fold_base["team_a_goals"],
    y_true_b=valid_fold_base["team_b_goals"],
    scoreline_prior=control_scoreline_prior,
    variant_name="control_static",
    decoder_grid=DECODER_GRID,
)

pred_a_control, pred_b_control = best_control_preds
valid_pred_match_control = build_prediction_frame(
    valid_fold_base,
    pred_a=pred_a_control,
    pred_b=pred_b_control,
    raw_outputs=control_raw_valid,
    variant_name="control_static",
)

valid_pred_match_control.to_csv(f"{PRED_DIR}/valid_pred_match_control.csv", index=False)
decoder_grid_control_df.to_csv(f"{SUM_DIR}/decoder_grid_control.csv", index=False)

control_valid_awmae = awmae_score(
    valid_fold_base["team_a_goals"], valid_fold_base["team_b_goals"],
    pred_a_control, pred_b_control,
    valid_fold_base["tournament"],
)

print(f"[OK] Control valid AW-MAE = {control_valid_awmae:.6f}")
display(decoder_grid_control_df.head(10))


tune_decoder_control_static:   0%|          | 0/144 [00:00<?, ?it/s]

[OK] Control valid AW-MAE = 3.108991


,variant_name,MAX_GOALS,w_direct,w_total,w_gd,w_outcome,w_prior,valid_awmae
0,control_static,6,0.5000,1.0000,1.5000,2.0000,0.5000,3.1090
1,control_static,8,0.5000,1.0000,1.5000,2.0000,0.5000,3.1105
2,control_static,7,0.5000,1.0000,1.5000,2.0000,0.5000,3.1108
3,control_static,6,1.0000,1.0000,1.5000,2.5000,0.5000,3.1140
4,control_static,6,0.5000,1.0000,1.5000,2.5000,0.5000,3.1146
5,control_static,8,1.0000,1.0000,1.5000,2.5000,0.5000,3.1157
6,control_static,7,1.0000,1.0000,1.5000,2.5000,0.5000,3.1158
7,control_static,8,0.5000,1.0000,1.5000,2.5000,0.5000,3.1162
8,control_static,7,0.5000,1.0000,1.5000,2.5000,0.5000,3.1164
9,control_static,6,0.5000,1.0000,1.5000,1.0000,0.5000,3.1201


Variant control ini adalah anchor pembanding internal di EXP 02.  
Jadi, kalau historical strength features memang berguna, **mereka harus bisa mengalahkan control ini secara fair** di validation yang sama.


## 11. Variant 1 — Historical Strength + Frozen Performance State


In [32]:
# Bangun static + history actual pre-match untuk train_fold
train_fold_hist_static = engineer_static_match_features(train_fold_hist)
train_fold_hist_train = train_fold_hist_static.copy()

history_models = train_catboost_decomposition_models(
    train_df=train_fold_hist_train,
    categorical_features=static_categorical_features,
    numeric_features=static_numeric_features + history_numeric_features,
    label_prefix="history",
)

print("[OK] History-augmented models selesai dilatih pada train_fold actual pre-match features.")


[INFO] Training history_goal_a ...
0:	learn: 1.7924579	total: 120ms	remaining: 2m 59s
200:	learn: 1.3876775	total: 26.5s	remaining: 2m 51s
400:	learn: 1.3323211	total: 52.4s	remaining: 2m 23s
600:	learn: 1.2882958	total: 1m 17s	remaining: 1m 56s
800:	learn: 1.2486255	total: 1m 44s	remaining: 1m 30s
1000:	learn: 1.2151883	total: 2m 10s	remaining: 1m 4s
1200:	learn: 1.1871606	total: 2m 36s	remaining: 38.9s
1400:	learn: 1.1589986	total: 3m 2s	remaining: 12.9s
1499:	learn: 1.1448048	total: 3m 15s	remaining: 0us
[INFO] Training history_goal_b ...
0:	learn: 1.7756973	total: 118ms	remaining: 2m 57s
200:	learn: 1.3807435	total: 26.1s	remaining: 2m 48s
400:	learn: 1.3271910	total: 51.5s	remaining: 2m 21s
600:	learn: 1.2858534	total: 1m 17s	remaining: 1m 56s
800:	learn: 1.2465936	total: 1m 44s	remaining: 1m 30s
1000:	learn: 1.2121560	total: 2m 10s	remaining: 1m 4s
1200:	learn: 1.1798272	total: 2m 36s	remaining: 39.1s
1400:	learn: 1.1488274	total: 3m 3s	remaining: 13s
1499:	learn: 1.1344534	total

In [33]:
def simulate_future_matches(
    future_match_df: pd.DataFrame,
    static_feature_df: pd.DataFrame,
    team_states_at_cutoff: dict,
    h2h_states_at_cutoff: dict,
    trained_models: dict,
    decoder_params: dict,
    mode: str = "freeze",
) -> pd.DataFrame:
    """Simulasi future matches secara kronologis.
    mode='freeze'    : hanya update last_match_date mengikuti jadwal.
    mode='recursive' : update full team/h2h state memakai prediksi skor final.
    """
    if mode not in {"freeze", "recursive"}:
        raise ValueError("mode harus 'freeze' atau 'recursive'")

    future_sorted = future_match_df.sort_values(["date", "match_id"]).reset_index(drop=True)
    static_sorted = static_feature_df.sort_values(["date", "match_id"]).reset_index(drop=True)
    if not np.array_equal(future_sorted["match_id"].values, static_sorted["match_id"].values):
        raise ValueError("future_match_df dan static_feature_df tidak aligned pada urutan match_id.")

    team_states = copy.deepcopy(team_states_at_cutoff)
    h2h_states = copy.deepcopy(h2h_states_at_cutoff)

    rows = []
    for match_row, static_row in tqdm(
        zip(future_sorted.to_dict("records"), static_sorted.to_dict("records")),
        total=len(future_sorted),
        desc=f"simulate_{mode}",
    ):
        match_series = pd.Series(match_row)
        hist_row = build_history_feature_row(match_series, team_states, h2h_states)
        full_row = {**static_row, **hist_row}
        feature_df = pd.DataFrame([full_row])

        raw_outputs = predict_decomposition_outputs(feature_df, trained_models)
        pred_a_int, pred_b_int = decode_batch_scores(
            pred_goal_a=raw_outputs["pred_goal_a_cont"],
            pred_goal_b=raw_outputs["pred_goal_b_cont"],
            pred_total=raw_outputs["pred_total_cont"],
            pred_gd=raw_outputs["pred_gd_cont"],
            pred_outcome_proba=raw_outputs["pred_outcome_proba"],
            scoreline_prior=decoder_params["scoreline_prior"],
            max_goals=int(decoder_params["MAX_GOALS"]),
            w_direct=float(decoder_params["w_direct"]),
            w_total=float(decoder_params["w_total"]),
            w_gd=float(decoder_params["w_gd"]),
            w_outcome=float(decoder_params["w_outcome"]),
            w_prior=float(decoder_params["w_prior"]),
        )

        pred_a_int = int(pred_a_int[0])
        pred_b_int = int(pred_b_int[0])

        out_row = {
            "match_id": match_row["match_id"],
            "date": match_row["date"],
            "gender": match_row["gender"],
            "tournament": match_row["tournament"],
            "neutral": match_row["neutral"],
            "team_a": match_row["team_a"],
            "team_b": match_row["team_b"],
            "pred_team_a_goals": pred_a_int,
            "pred_team_b_goals": pred_b_int,
            "pred_goal_a_cont": float(raw_outputs["pred_goal_a_cont"][0]),
            "pred_goal_b_cont": float(raw_outputs["pred_goal_b_cont"][0]),
            "pred_total_cont": float(raw_outputs["pred_total_cont"][0]),
            "pred_gd_cont": float(raw_outputs["pred_gd_cont"][0]),
            "pred_outcome_team_a_win": float(raw_outputs["pred_outcome_proba"][0, 0]),
            "pred_outcome_draw": float(raw_outputs["pred_outcome_proba"][0, 1]),
            "pred_outcome_team_b_win": float(raw_outputs["pred_outcome_proba"][0, 2]),
            "variant_name": f"history_{mode}",
        }

        if "team_a_goals" in match_row and "team_b_goals" in match_row:
            out_row["actual_team_a_goals"] = int(match_row["team_a_goals"])
            out_row["actual_team_b_goals"] = int(match_row["team_b_goals"])
            out_row["awmae_match_loss"] = official_match_loss(
                out_row["actual_team_a_goals"],
                out_row["actual_team_b_goals"],
                pred_a_int,
                pred_b_int,
            )

        rows.append(out_row)

        gender = str(match_row["gender"])
        team_a = str(match_row["team_a"])
        team_b = str(match_row["team_b"])
        key_a = _team_state_key(gender, team_a)
        key_b = _team_state_key(gender, team_b)
        h2h_key = _canonical_pair_key(gender, team_a, team_b)

        if mode == "freeze":
            # Hanya gerakkan last_match_date; performance state tetap beku.
            state_a = copy.deepcopy(team_states.get(key_a, init_team_state()))
            state_b = copy.deepcopy(team_states.get(key_b, init_team_state()))
            state_a["last_match_date"] = pd.Timestamp(match_row["date"])
            state_b["last_match_date"] = pd.Timestamp(match_row["date"])
            team_states[key_a] = state_a
            team_states[key_b] = state_b
        else:
            state_a = copy.deepcopy(team_states.get(key_a, init_team_state()))
            state_b = copy.deepcopy(team_states.get(key_b, init_team_state()))
            h2h_state = copy.deepcopy(h2h_states.get(h2h_key, init_h2h_state()))

            match_context = {
                "date": match_row["date"],
                "neutral": match_row.get("neutral", 0),
                "team_a_is_home": match_row.get("team_a_is_home", 0),
                "team_b_is_home": match_row.get("team_b_is_home", 0),
                "tournament": match_row.get("tournament", ""),
            }
            state_a, state_b, h2h_state = update_states_from_score(
                state_a, state_b, h2h_state,
                match_context=match_context,
                goals_a=pred_a_int,
                goals_b=pred_b_int,
            )
            team_states[key_a] = state_a
            team_states[key_b] = state_b
            h2h_states[h2h_key] = h2h_state

    return pd.DataFrame(rows)

def tune_decoder_via_simulation(
    future_match_df: pd.DataFrame,
    static_feature_df: pd.DataFrame,
    team_states_at_cutoff: dict,
    h2h_states_at_cutoff: dict,
    trained_models: dict,
    decoder_grid: dict,
    scoreline_prior: dict,
    mode: str,
):
    combos = list(product(
        decoder_grid["MAX_GOALS"],
        decoder_grid["w_direct"],
        decoder_grid["w_total"],
        decoder_grid["w_gd"],
        decoder_grid["w_outcome"],
        decoder_grid["w_prior"],
    ))

    rows = []
    best_score = np.inf
    best_params = None
    best_pred_df = None

    for max_goals, w_direct, w_total, w_gd, w_outcome, w_prior in tqdm(combos, desc=f"tune_sim_{mode}"):
        decoder_params = {
            "MAX_GOALS": int(max_goals),
            "w_direct": float(w_direct),
            "w_total": float(w_total),
            "w_gd": float(w_gd),
            "w_outcome": float(w_outcome),
            "w_prior": float(w_prior),
            "scoreline_prior": scoreline_prior,
        }
        pred_df = simulate_future_matches(
            future_match_df=future_match_df,
            static_feature_df=static_feature_df,
            team_states_at_cutoff=team_states_at_cutoff,
            h2h_states_at_cutoff=h2h_states_at_cutoff,
            trained_models=trained_models,
            decoder_params=decoder_params,
            mode=mode,
        )
        score = awmae_score(
            pred_df["actual_team_a_goals"],
            pred_df["actual_team_b_goals"],
            pred_df["pred_team_a_goals"],
            pred_df["pred_team_b_goals"],
            pred_df["tournament"],
        )
        row = {
            "variant_name": f"history_{mode}",
            "MAX_GOALS": int(max_goals),
            "w_direct": float(w_direct),
            "w_total": float(w_total),
            "w_gd": float(w_gd),
            "w_outcome": float(w_outcome),
            "w_prior": float(w_prior),
            "valid_awmae": float(score),
        }
        rows.append(row)

        if score < best_score:
            best_score = float(score)
            best_params = row.copy()
            best_pred_df = pred_df.copy()

    grid_df = pd.DataFrame(rows).sort_values("valid_awmae").reset_index(drop=True)
    return grid_df, best_params, best_pred_df

print("[OK] simulate_future_matches() dan tuning by simulation siap.")


[OK] simulate_future_matches() dan tuning by simulation siap.


In [34]:
history_scoreline_prior = build_scoreline_prior(
    train_fold_base["team_a_goals"].astype(int),
    train_fold_base["team_b_goals"].astype(int),
    max_goals=max(DECODER_GRID["MAX_GOALS"]),
    alpha=1.0,
)

decoder_grid_frozen_df, best_frozen_params, valid_pred_match_frozen = tune_decoder_via_simulation(
    future_match_df=valid_fold_base,
    static_feature_df=valid_fold_static,
    team_states_at_cutoff=cutoff_team_states_valid,
    h2h_states_at_cutoff=cutoff_h2h_states_valid,
    trained_models=history_models,
    decoder_grid=DECODER_GRID,
    scoreline_prior=history_scoreline_prior,
    mode="freeze",
)

valid_pred_match_frozen.to_csv(f"{PRED_DIR}/valid_pred_match_frozen.csv", index=False)
decoder_grid_frozen_df.to_csv(f"{SUM_DIR}/decoder_grid_frozen.csv", index=False)

frozen_valid_awmae = awmae_score(
    valid_pred_match_frozen["actual_team_a_goals"],
    valid_pred_match_frozen["actual_team_b_goals"],
    valid_pred_match_frozen["pred_team_a_goals"],
    valid_pred_match_frozen["pred_team_b_goals"],
    valid_pred_match_frozen["tournament"],
)

print(f"[OK] history_frozen valid AW-MAE = {frozen_valid_awmae:.6f}")
display(decoder_grid_frozen_df.head(10))


tune_sim_freeze:   0%|          | 0/144 [00:00<?, ?it/s]

simulate_freeze:   0%|          | 0/7877 [00:00<?, ?it/s]

simulate_freeze:   0%|          | 0/7877 [00:00<?, ?it/s]

simulate_freeze:   0%|          | 0/7877 [00:00<?, ?it/s]

simulate_freeze:   0%|          | 0/7877 [00:00<?, ?it/s]

simulate_freeze:   0%|          | 0/7877 [00:00<?, ?it/s]

simulate_freeze:   0%|          | 0/7877 [00:00<?, ?it/s]

KeyboardInterrupt: 

Mode frozen adalah mode paling konservatif.  
State performa tim dianggap **beku sejak cutoff**, tetapi `last_match_date` tetap bergerak mengikuti jadwal, sehingga rest-day features masih bisa berubah secara legal.


## 12. Variant 2 — Historical Strength + Recursive Pseudo-Update


In [ ]:
decoder_grid_recursive_df, best_recursive_params, valid_pred_match_recursive = tune_decoder_via_simulation(
    future_match_df=valid_fold_base,
    static_feature_df=valid_fold_static,
    team_states_at_cutoff=cutoff_team_states_valid,
    h2h_states_at_cutoff=cutoff_h2h_states_valid,
    trained_models=history_models,
    decoder_grid=DECODER_GRID,
    scoreline_prior=history_scoreline_prior,
    mode="recursive",
)

valid_pred_match_recursive.to_csv(f"{PRED_DIR}/valid_pred_match_recursive.csv", index=False)
decoder_grid_recursive_df.to_csv(f"{SUM_DIR}/decoder_grid_recursive.csv", index=False)

recursive_valid_awmae = awmae_score(
    valid_pred_match_recursive["actual_team_a_goals"],
    valid_pred_match_recursive["actual_team_b_goals"],
    valid_pred_match_recursive["pred_team_a_goals"],
    valid_pred_match_recursive["pred_team_b_goals"],
    valid_pred_match_recursive["tournament"],
)

print(f"[OK] history_recursive valid AW-MAE = {recursive_valid_awmae:.6f}")
display(decoder_grid_recursive_df.head(10))


Mode recursive lebih adaptif karena state masa depan benar-benar berevolusi.  
Namun trade-off utamanya juga jelas: **kalau prediksi awal salah, error itu bisa merambat ke match berikutnya**.


## 13. Perbandingan Hasil antar Variant


In [ ]:
variant_results = pd.DataFrame([
    {
        "variant_name": "control_static",
        "valid_awmae": float(control_valid_awmae),
        "decoder_best_params": json.dumps({
            k: v for k, v in best_control_params.items()
            if k in {"MAX_GOALS", "w_direct", "w_total", "w_gd", "w_outcome", "w_prior", "valid_awmae"}
        }),
        "notes": "Static shared features only (EXP 01 core).",
    },
    {
        "variant_name": "history_frozen",
        "valid_awmae": float(frozen_valid_awmae),
        "decoder_best_params": json.dumps({
            k: v for k, v in best_frozen_params.items()
            if k in {"MAX_GOALS", "w_direct", "w_total", "w_gd", "w_outcome", "w_prior", "valid_awmae"}
        }),
        "notes": "Static + history, state performance dibekukan.",
    },
    {
        "variant_name": "history_recursive",
        "valid_awmae": float(recursive_valid_awmae),
        "decoder_best_params": json.dumps({
            k: v for k, v in best_recursive_params.items()
            if k in {"MAX_GOALS", "w_direct", "w_total", "w_gd", "w_outcome", "w_prior", "valid_awmae"}
        }),
        "notes": "Static + history, state di-update pakai prediksi sendiri.",
    },
]).sort_values("valid_awmae").reset_index(drop=True)

display(variant_results)


In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=variant_results, x="variant_name", y="valid_awmae")
plt.title("Validation AW-MAE Comparison Across Variants")
plt.xlabel("Variant")
plt.ylabel("AW-MAE")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/valid_awmae_variant_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
compare_cols = ["match_id", "date", "tournament", "team_a", "team_b", "actual_team_a_goals", "actual_team_b_goals",
                "pred_team_a_goals", "pred_team_b_goals", "awmae_match_loss"]

comparison_sample = (
    valid_pred_match_control[compare_cols]
    .rename(columns={
        "pred_team_a_goals": "pred_a_control",
        "pred_team_b_goals": "pred_b_control",
        "awmae_match_loss": "loss_control",
    })
    .merge(
        valid_pred_match_frozen[["match_id", "pred_team_a_goals", "pred_team_b_goals", "awmae_match_loss"]].rename(columns={
            "pred_team_a_goals": "pred_a_frozen",
            "pred_team_b_goals": "pred_b_frozen",
            "awmae_match_loss": "loss_frozen",
        }),
        on="match_id", how="left",
    )
    .merge(
        valid_pred_match_recursive[["match_id", "pred_team_a_goals", "pred_team_b_goals", "awmae_match_loss"]].rename(columns={
            "pred_team_a_goals": "pred_a_recursive",
            "pred_team_b_goals": "pred_b_recursive",
            "awmae_match_loss": "loss_recursive",
        }),
        on="match_id", how="left",
    )
)

display(comparison_sample.head(15))


Bagian ini adalah checkpoint utama eksperimen:

- apakah historical features benar-benar membantu,
- apakah frozen cukup, atau recursive lebih kuat,
- dan apakah gain yang didapat cukup stabil untuk dibawa ke test inference.


## 14. Feature Importance Shift Analysis


In [ ]:
def get_feature_importance_frame(model, feature_names, top_k=20):
    imp = pd.DataFrame({
        "feature": feature_names,
        "importance": model.get_feature_importance(),
    }).sort_values("importance", ascending=False).reset_index(drop=True)
    return imp.head(top_k)

control_outcome_imp = get_feature_importance_frame(
    control_models["model_outcome"],
    control_models["feature_cols"],
    top_k=20,
)
history_outcome_imp = get_feature_importance_frame(
    history_models["model_outcome"],
    history_models["feature_cols"],
    top_k=20,
)

display(control_outcome_imp)
display(history_outcome_imp)


In [ ]:
plt.figure(figsize=(8, 6))
sns.barplot(data=control_outcome_imp.head(15), y="feature", x="importance")
plt.title("Top Feature Importance — Outcome Control")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/top_feature_importance_outcome_control.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 6))
sns.barplot(data=history_outcome_imp.head(15), y="feature", x="importance")
plt.title("Top Feature Importance — Outcome History")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/top_feature_importance_outcome_history.png", dpi=150, bbox_inches="tight")
plt.show()


Tujuan utama analisis ini adalah melihat apakah ketika history ditambahkan, dominasi fitur identitas mentah seperti `team_a`, `team_b`, `pair_key`, atau `match_year` mulai **bergeser** ke arah fitur strength/history yang lebih bermakna.


## 15. Error Analysis per Subgroup


In [ ]:
def summarize_subgroup_awmae(pred_df: pd.DataFrame, group_col: str, variant_name: str) -> pd.DataFrame:
    rows = []
    for key, sub in pred_df.groupby(group_col, dropna=False):
        score = awmae_score(
            sub["actual_team_a_goals"],
            sub["actual_team_b_goals"],
            sub["pred_team_a_goals"],
            sub["pred_team_b_goals"],
            sub["tournament"],
        )
        rows.append({
            "variant_name": variant_name,
            "group_col": group_col,
            "group_value": str(key),
            "n_matches": len(sub),
            "awmae": score,
        })
    return pd.DataFrame(rows).sort_values("awmae").reset_index(drop=True)

def add_seen_unseen_flag(pred_df: pd.DataFrame, seen_teams: set) -> pd.DataFrame:
    df = pred_df.copy()
    df["unseen_context"] = np.where(
        (~df["team_a"].isin(seen_teams)) | (~df["team_b"].isin(seen_teams)),
        "unseen_team_present",
        "seen_only",
    )
    return df

def add_extreme_flag(pred_df: pd.DataFrame, threshold: int = 6) -> pd.DataFrame:
    df = pred_df.copy()
    max_actual = df[["actual_team_a_goals", "actual_team_b_goals"]].max(axis=1)
    df["scoreline_bucket"] = np.where(max_actual >= threshold, "extreme", "non_extreme")
    return df

seen_teams_train_fold = set(train_fold_base["team_a"]).union(set(train_fold_base["team_b"]))

control_eval_df = add_extreme_flag(add_seen_unseen_flag(valid_pred_match_control, seen_teams_train_fold))
frozen_eval_df = add_extreme_flag(add_seen_unseen_flag(valid_pred_match_frozen, seen_teams_train_fold))
recursive_eval_df = add_extreme_flag(add_seen_unseen_flag(valid_pred_match_recursive, seen_teams_train_fold))

subgroup_gender_df = pd.concat([
    summarize_subgroup_awmae(control_eval_df, "gender", "control_static"),
    summarize_subgroup_awmae(frozen_eval_df, "gender", "history_frozen"),
    summarize_subgroup_awmae(recursive_eval_df, "gender", "history_recursive"),
], ignore_index=True)

subgroup_neutral_df = pd.concat([
    summarize_subgroup_awmae(control_eval_df, "neutral", "control_static"),
    summarize_subgroup_awmae(frozen_eval_df, "neutral", "history_frozen"),
    summarize_subgroup_awmae(recursive_eval_df, "neutral", "history_recursive"),
], ignore_index=True)

subgroup_extreme_df = pd.concat([
    summarize_subgroup_awmae(control_eval_df, "scoreline_bucket", "control_static"),
    summarize_subgroup_awmae(frozen_eval_df, "scoreline_bucket", "history_frozen"),
    summarize_subgroup_awmae(recursive_eval_df, "scoreline_bucket", "history_recursive"),
], ignore_index=True)

display(subgroup_gender_df)
display(subgroup_neutral_df)
display(subgroup_extreme_df)


In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=subgroup_gender_df, x="group_value", y="awmae", hue="variant_name")
plt.title("Subgroup AW-MAE by Gender")
plt.xlabel("Gender")
plt.ylabel("AW-MAE")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/subgroup_awmae_by_gender.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 4))
sns.barplot(data=subgroup_neutral_df, x="group_value", y="awmae", hue="variant_name")
plt.title("Subgroup AW-MAE by Neutral")
plt.xlabel("Neutral")
plt.ylabel("AW-MAE")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/subgroup_awmae_by_neutral.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 4))
sns.barplot(data=subgroup_extreme_df, x="group_value", y="awmae", hue="variant_name")
plt.title("Subgroup AW-MAE by Extreme vs Non-Extreme Scoreline")
plt.xlabel("Scoreline Bucket")
plt.ylabel("AW-MAE")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/scoreline_extreme_error_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


Di tahap ini yang dicari bukan hanya “siapa yang paling kecil AW-MAE totalnya”, tetapi juga:

- apakah women matches membaik,
- apakah neutral matches membaik,
- apakah history membantu di scoreline ekstrem,
- dan apakah recursive update memberi manfaat nyata atau justru menambah propagation error.


## 16. Pilih Pipeline Terbaik dan Retrain di Full Train


In [ ]:
best_variant_name = variant_results.iloc[0]["variant_name"]
print(f"[INFO] Best variant dari validation: {best_variant_name}")

best_decoder_lookup = {
    "control_static": best_control_params,
    "history_frozen": best_frozen_params,
    "history_recursive": best_recursive_params,
}
best_decoder_params = best_decoder_lookup[best_variant_name].copy()

# Full train static
train_full_static = engineer_static_match_features(train_match_base)
test_full_static = engineer_static_match_features(test_match_base)

# Full-train history actual pre-match
train_full_hist_actual, full_team_states_cutoff, full_h2h_states_cutoff = build_train_history_features(train_match_base)
train_full_hist_df = train_match_base.merge(train_full_hist_actual, on="match_id", how="left", validate="one_to_one")
train_full_hist_static = engineer_static_match_features(train_full_hist_df)

full_scoreline_prior = build_scoreline_prior(
    train_match_base["team_a_goals"].astype(int),
    train_match_base["team_b_goals"].astype(int),
    max_goals=max(DECODER_GRID["MAX_GOALS"]),
    alpha=1.0,
)
best_decoder_params["scoreline_prior"] = full_scoreline_prior

print("[OK] Full-train data, full history features, dan full scoreline prior siap.")


In [ ]:
# Retrain sesuai variant terbaik
full_control_models = None
full_history_models = None

if best_variant_name == "control_static":
    full_control_models = train_catboost_decomposition_models(
        train_df=train_full_static,
        categorical_features=static_categorical_features,
        numeric_features=static_numeric_features,
        label_prefix="full_control",
    )
else:
    full_history_models = train_catboost_decomposition_models(
        train_df=train_full_hist_static,
        categorical_features=static_categorical_features,
        numeric_features=static_numeric_features + history_numeric_features,
        label_prefix="full_history",
    )

print("[OK] Retrain pipeline terbaik selesai.")


Retrain full-train dilakukan **setelah** best variant diputuskan di validation.  
Decoder best params dari validation dipakai **apa adanya**, hanya prior scoreline-nya yang dibangun ulang dari full train karena itu memang bagian dari data training final.


## 17. Inference pada Test Match Secara Kronologis


In [ ]:
if best_variant_name == "control_static":
    test_raw_outputs = predict_decomposition_outputs(test_full_static, full_control_models)
    test_pred_a, test_pred_b = decode_batch_scores(
        pred_goal_a=test_raw_outputs["pred_goal_a_cont"],
        pred_goal_b=test_raw_outputs["pred_goal_b_cont"],
        pred_total=test_raw_outputs["pred_total_cont"],
        pred_gd=test_raw_outputs["pred_gd_cont"],
        pred_outcome_proba=test_raw_outputs["pred_outcome_proba"],
        scoreline_prior=best_decoder_params["scoreline_prior"],
        max_goals=int(best_decoder_params["MAX_GOALS"]),
        w_direct=float(best_decoder_params["w_direct"]),
        w_total=float(best_decoder_params["w_total"]),
        w_gd=float(best_decoder_params["w_gd"]),
        w_outcome=float(best_decoder_params["w_outcome"]),
        w_prior=float(best_decoder_params["w_prior"]),
    )
    test_pred_match_best = build_prediction_frame(
        test_match_base,
        pred_a=test_pred_a,
        pred_b=test_pred_b,
        raw_outputs=test_raw_outputs,
        variant_name=best_variant_name,
    )
else:
    sim_mode = "freeze" if best_variant_name == "history_frozen" else "recursive"
    test_pred_match_best = simulate_future_matches(
        future_match_df=test_match_base,
        static_feature_df=test_full_static,
        team_states_at_cutoff=full_team_states_cutoff,
        h2h_states_at_cutoff=full_h2h_states_cutoff,
        trained_models=full_history_models,
        decoder_params=best_decoder_params,
        mode=sim_mode,
    )

test_pred_match_best.to_csv(f"{PRED_DIR}/test_pred_match_best.csv", index=False)
print(f"[OK] Test match-level prediction disimpan ke {PRED_DIR}/test_pred_match_best.csv")
display(test_pred_match_best.head(10))


Pada test tidak ada target aktual.  
Karena itu seluruh update state di test hanya boleh berasal dari:

- informasi jadwal yang memang diketahui,
- dan, bila mode terbaik adalah recursive, dari **prediksi model sendiri**.


## 18. Reverse Mapping ke Submission


In [ ]:
submission_exp02 = match_predictions_to_submission(
    test_row_df=test,
    pred_match_df=test_pred_match_best,
    canonical_team_a_col="team_a",
    canonical_team_b_col="team_b",
    pred_a_col="pred_team_a_goals",
    pred_b_col="pred_team_b_goals",
)

assert submission_exp02.shape[0] == sample_sub.shape[0], "Jumlah row submission tidak sama dengan sample submission"
assert list(submission_exp02.columns) == ["Id", "team_goals", "opp_goals"], "Format kolom submission salah"
assert submission_exp02["Id"].astype(str).tolist() == sample_sub["Id"].astype(str).tolist(), "Urutan Id submission tidak sama dengan sample submission"
assert submission_exp02.isna().sum().sum() == 0, "Masih ada missing prediction pada submission"

submission_exp02.to_csv(f"{SUB_DIR}/submission_exp02_best.csv", index=False)
print(f"[OK] Submission final disimpan ke {SUB_DIR}/submission_exp02_best.csv")
display(submission_exp02.head(10))


## 19. Ringkasan Hasil Eksperimen


In [ ]:
metrics_payload = {
    "control_static_valid_awmae": float(control_valid_awmae),
    "history_frozen_valid_awmae": float(frozen_valid_awmae),
    "history_recursive_valid_awmae": float(recursive_valid_awmae),
    "best_variant_name": best_variant_name,
    "best_decoder_params": {
        k: (float(v) if isinstance(v, (int, float, np.integer, np.floating)) else v)
        for k, v in best_decoder_params.items()
        if k != "scoreline_prior"
    },
    "n_train_matches": int(len(train_match_base)),
    "n_test_matches": int(len(test_match_base)),
    "n_train_rows": int(len(train)),
    "n_test_rows": int(len(test)),
}

with open(f"{SUM_DIR}/exp02_metrics.json", "w") as f:
    json.dump(metrics_payload, f, indent=2)

notes = [
    "EXP 02 summary",
    f"- control_static valid AW-MAE    : {control_valid_awmae:.6f}",
    f"- history_frozen valid AW-MAE   : {frozen_valid_awmae:.6f}",
    f"- history_recursive valid AW-MAE: {recursive_valid_awmae:.6f}",
    f"- best variant                  : {best_variant_name}",
    "",
    "Catatan:",
    "- historical features dibangun pre-match secara leakage-safe",
    "- validation future simulation tidak memakai actual valid outcomes untuk update state",
    "- test simulation sepenuhnya legal: hanya schedule-known info dan prediksi model sendiri",
]
with open(f"{SUM_DIR}/experiment_notes.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(notes))

print(f"[OK] Metrics disimpan ke {SUM_DIR}/exp02_metrics.json")
print(f"[OK] Notes disimpan ke {SUM_DIR}/experiment_notes.txt")
print(json.dumps(metrics_payload, indent=2))


Secara objektif, ringkasan yang perlu dilihat dari EXP 02 adalah:

- variant mana yang paling baik di validation,
- apakah historical features benar-benar membantu dibanding control,
- apakah recursive update membantu atau justru membawa error propagation,
- apakah feature importance bergeser ke arah strength/history,
- dan subgroup mana yang masih sulit.


## 20. Next Step ke EXP 03

Eksperimen berikutnya bisa bergerak ke salah satu dari dua arah utama:

1. **modeling score distribution yang lebih probabilistik**, agar decoder punya sinyal yang lebih kaya daripada hanya direct/total/gd/outcome;
2. **pemanfaatan train-only signal secara aman** melalui skema teacher-student atau distillation, tanpa melanggar legalitas kompetisi saat inference test.

Dengan begitu, EXP 03 bisa fokus pada peningkatan kualitas distribusi skor atau transfer strength signal, sambil tetap menjaga pipeline tetap feasible di test.
